In [0]:

from pyspark.sql import functions as F

BRZ, SLV, GLD = "bakehouse_project.brz", "bakehouse_project.slv", "bakehouse_project.gld"
META = "bakehouse_project.p_metadata"

rows = spark.sql(f"select batch_id from {META}.batch_control where batch_status='opened'").collect()
if not rows:
    raise Exception("No opened batch found.")
batch_id = str(rows[0]["batch_id"])


# ---------- helpers ----------
def n_rows(t, where=None):
    df = spark.read.table(t)
    return (df.filter(where) if where else df).count()

def n_batch(t, where=None):
    df = spark.read.table(t).filter(F.col("batch_id").cast("string") == batch_id)
    return (df.filter(where) if where else df).count()

def n_keys(t, key):
    return spark.read.table(t).where(F.col(key).isNotNull()).select(key).distinct().count()

def pk_issues(t, pks):
    nulls = dups = 0
    for keys, where in pks:
        df = spark.read.table(t)
        if where:
            df = df.filter(where)
        nulls += df.filter(" OR ".join(f"`{k}` IS NULL" for k in keys)).count()
        dups += df.groupBy(*keys).count().filter("count > 1").count()
    return nulls, dups

# ---------- checks ----------
# (source_table, target_table, source count, target count, table for PK checks, PK checks, fail job on count mismatch?)
CUR_S = "`__END_AT` IS NULL"
CUR_G = "end_at IS NULL"

CHECKS = [
    # customers (SCD1)
    ("b_customers", "s_customers",
     lambda: n_keys(f"{BRZ}.b_customers", "customerID"),
     lambda: n_rows(f"{SLV}.s_customers"),
     f"{SLV}.s_customers", [(["customer_id"], None)], True),

    ("s_customers", "dim_customer",
     lambda: n_rows(f"{SLV}.s_customers"),
     lambda: n_rows(f"{GLD}.dim_customer"),
     f"{GLD}.dim_customer", [(["customer_id"], None), (["customer_key"], None)], True),

    # suppliers (SCD2)
    ("b_suppliers", "s_suppliers",
     lambda: n_keys(f"{BRZ}.b_suppliers", "supplierID"),
     lambda: n_rows(f"{SLV}.s_suppliers", CUR_S),
     f"{SLV}.s_suppliers", [(["supplier_id"], CUR_S)], True),

    ("s_suppliers", "dim_supplier",
     lambda: n_rows(f"{SLV}.s_suppliers"),
     lambda: n_rows(f"{GLD}.dim_supplier"),
     f"{GLD}.dim_supplier", [(["supplier_id"], CUR_G), (["supplier_key"], None)], True),

    # franchises (SCD2)
    ("b_franchises", "s_franchises",
     lambda: n_keys(f"{BRZ}.b_franchises", "franchiseID"),
     lambda: n_rows(f"{SLV}.s_franchises", CUR_S),
     f"{SLV}.s_franchises", [(["franchise_id"], CUR_S)], True),

    ("s_franchises", "dim_franchise",
     lambda: n_rows(f"{SLV}.s_franchises"),
     lambda: n_rows(f"{GLD}.dim_franchise"),
     f"{GLD}.dim_franchise", [(["franchise_id"], CUR_G), (["franchise_key"], None)], True),

    # transactions (append)
    ("b_transactions", "s_transactions",
     lambda: n_batch(f"{BRZ}.b_transactions"),
     lambda: n_batch(f"{SLV}.s_transactions"),
     f"{SLV}.s_transactions", [(["transaction_id"], None)], False),

    ("s_transactions", "fact_sales",
     lambda: n_batch(f"{SLV}.s_transactions"),
     lambda: n_batch(f"{GLD}.fact_sales"),
     f"{GLD}.fact_sales", [(["transaction_id"], None)], False),

    # reviews
    ("b_media_customer_reviews", "s_reviews",
     lambda: n_batch(f"{BRZ}.b_media_customer_reviews"),
     lambda: n_batch(f"{SLV}.s_reviews", "brz_table = 'b_media_customer_reviews'"),
     f"{SLV}.s_reviews", [], False),

    ("b_media_gold_reviews_chunked", "s_reviews",
     lambda: n_batch(f"{BRZ}.b_media_gold_reviews_chunked"),
     lambda: n_batch(f"{SLV}.s_reviews", "brz_table = 'b_media_gold_reviews_chunked'"),
     f"{SLV}.s_reviews", [], False),

    ("s_reviews", "fact_reviews",
     lambda: n_batch(f"{SLV}.s_reviews"),
     lambda: n_batch(f"{GLD}.fact_reviews"),
     f"{GLD}.fact_reviews", [], False),
]

# ---------- run ----------
out, failed = [], []

for source_table, target_table, src_fn, tgt_fn, pk_table, pks, strict_count in CHECKS:
    src, tgt = src_fn(), tgt_fn()
    nulls, dups = pk_issues(pk_table, pks)

    issues = []
    if src != tgt:
        issues.append("mismatch")
    if nulls:
        issues.append("null_pk")
    if dups:
        issues.append("duplicate_pk")

    out.append((
        batch_id, source_table, target_table,
        src, tgt, src - tgt, nulls, dups,
        ", ".join(issues) if issues else "match"
    ))

    if nulls or dups or (src != tgt and strict_count):
        failed.append(f"{source_table} -> {target_table}")

df = spark.createDataFrame(
    out,
    "batch_id string, source_table string, target_table string, "
    "source_rowcount long, target_rowcount long, diff long, "
    "null_pk_count long, duplicate_pk_count long, msg string"
)

df = df.withColumn("date", F.current_timestamp())

df.write.mode("append").saveAsTable(f"{META}.dq_checks")

if failed:
    raise Exception(f"Audit failed for: {failed}")
